# NVDA Data Center 数值来源：已完成 Recorder 的输出收尾

只读同一 FINISHED Recorder 的三个已保存对象，规范化 JSON 标量并归档；不重读 SEC 原件或新建 Recorder。


In [1]:
import os,sys,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings
card=root/'research/stage1_nvda_datacenter_original_numeric_20261023';p=json.loads((card/'output_finalize_plan.json').read_text());f=json.loads((card/'attempt1_serialization_failure.json').read_text());sha=lambda x:hashlib.sha256(Path(x).read_bytes()).hexdigest()
assert p['frozen'] and p['parent_plan_sha256']==sha(card/'plan.json') and p['recorder_id']==f['recorder_id'] and p['experiment_id']==f['experiment_id']
assert all(p['budget'][k]==0 for k in ('new_HTTP_GETs','new_raw_original_reads','new_Qlib_Recorders','new_model_fits','new_native_accounts'))
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(card/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=p['recorder_id'],experiment_id=p['experiment_id']);assert rec.status=='FINISHED'
 saved=rec.load_object('analysis/report');events=rec.load_object('analysis/event_rows');frame=rec.load_object('analysis/source_frame')
assert len(events)==63 and len(frame)==3966 and saved['plan_sha256']==sha(card/'plan.json') and saved['selected_events']==63
assert saved['numeric_events']==sum(e['status']=='numeric_same_release_table' for e in events)
assert saved['training_rows']==3020 and saved['held_rows']==946
normalize=lambda o:int(o) if isinstance(o,np.integer) else float(o) if isinstance(o,np.floating) else bool(o) if isinstance(o,np.bool_) else o.isoformat() if isinstance(o,(pd.Timestamp,)) else str(o)
result=json.loads(json.dumps({**saved,'events':events},default=normalize))
assert result['unknown_by_status']=={'no_exact_Data_Center_market_platform_row':21,'ambiguous_multiple_Data_Center_rows':10}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':json.loads((card/'plan.json').read_text())['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'recorder_id':p['recorder_id'],'experiment_id':p['experiment_id'],'recorder_uri':str(card/'qlib_recorder'),'result':'result.json','numeric_events':result['numeric_events'],'training_known_days':result['training_known_days'],'held_known_days':result['held_known_days'],'model_fits':0,'native_accounts':0,'artifact_policy':json.loads((card/'plan.json').read_text())['artifact_policy'],'output_finalize':'output_finalize_receipt.json','limitations':['Same-release CFO table values are source evidence but 10 annual quarterly rows remain ambiguous and 21 events lack exact rows.','SEC acceptance next-Qlib-day is conservative, but issuer first-public release clock and platform-definition continuity remain unqualified.','No SOXX future outcomes, information fit, fee-net account or forward validation.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':summary['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=p['recorder_id'],experiment_id=p['experiment_id'])
with tempfile.TemporaryDirectory(prefix='nvda-dc-finalize-') as tmp:
 archive=Path(tmp)/'recorder.tar.gz'
 with archive.open('wb') as out:proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert proc.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/report') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
receipt={'status':'same_finished_Recorder_output_only_finalized','parent_plan_sha256':p['parent_plan_sha256'],'finalize_plan_sha256':sha(card/'output_finalize_plan.json'),'recorder_id':p['recorder_id'],'experiment_id':p['experiment_id'],'saved_event_rows':len(events),'saved_source_frame_rows':len(frame),'numeric_events':result['numeric_events'],'result_sha256':sha(card/'result.json'),'NAS_recorder_sha256':e['sha256'],'new_raw_reads':0,'new_Qlib_Recorders':0,'new_model_fits':0,'new_native_accounts':0}
(card/'output_finalize_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
print(json.dumps({'numeric_events':result['numeric_events'],'unknown':result['unknown_by_status'],'train_known':result['training_known_days'],'held_known':result['held_known_days'],'recorder_id':p['recorder_id'],'nas_sha256':e['sha256']},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[28629:MainThread](2026-10-23 08:14:10,924) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[28629:MainThread](2026-10-23 08:14:10,926) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[28629:MainThread](2026-10-23 08:14:10,926) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{
  "numeric_events": 32,
  "unknown": {
    "no_exact_Data_Center_market_platform_row": 21,
    "ambiguous_multiple_Data_Center_rows": 10
  },
  "train_known": 1284,
  "held_known": 696,
  "recorder_id": "79535324617845bc9c5f75bb60c297b6",
  "nas_sha256": "674874c024ad7ece4c22a5d7b5700de0ed480bbda6e2c0ffefa39c9e723d28d5"
}
